In [7]:
import cloudinary
import cloudinary.uploader
import json

In [8]:
cloudinary.config(
    cloud_name="dozpywcus",
    api_key="345466748819321",  # Điền API Key của bạn vào đây
    api_secret="hX6w0WmgPjD1ffxaDXs69QUjiDQ",  # Điền API Secret của bạn vào đây
    secure=True,
)

In [18]:

def upload_to_cloudinary(source_url, public_id=None):
    try:
        print("Đang upload file lên Cloudinary...")

        # 3. Tiến hành upload
        # Bạn có thể đổi 'public_id' thành tên file bạn muốn lưu trên Cloud
        upload_result = cloudinary.uploader.upload(
            source_url, public_id=public_id
        )

        # 4. Lấy link Cloudinary trả về
        # Sử dụng 'secure_url' để lấy link https an toàn
        cloudinary_url = upload_result.get("secure_url")

        print("\n--- Upload thành công! ---")
        print(f"Link Cloudinary của bạn: {cloudinary_url}")
        return cloudinary_url

    except Exception as e:
        print(f"Đã xảy ra lỗi khi upload: {e}")

In [ ]:
source_url = "https://cdn.hoanghamobile.vn//Uploads/2026/04/20/xiaomi-note-14-web.jpg"
public_id= "banners"
url_cloudinary = upload_to_cloudinary(source_url, public_id)
print(url_cloudinary)

Đang upload file lên Cloudinary...

--- Upload thành công! ---
Link Cloudinary của bạn: https://res.cloudinary.com/dozpywcus/image/upload/v1780382349/my_uploaded_python_logo.jpg
https://res.cloudinary.com/dozpywcus/image/upload/v1780382349/my_uploaded_python_logo.jpg


In [19]:
public_id= "banners"
with open("banner.json", "r", encoding="utf-8") as file:
    data = json.load(file)
print("Dữ liệu đã đọc từ banner.json:")
print(data[0].get("image_url"))

base_public_id = "banners"
for i, item in enumerate(data):
    image_url = item.get("image_url")
    if image_url:
        unique_public_id = f"{base_public_id}_{i+1}"
        cloudinary_url = upload_to_cloudinary(image_url, unique_public_id)
        if cloudinary_url:
            item["image_url"] = cloudinary_url

# Lưu dữ liệu đã cập nhật vào file JSON
with open("banner_updated.json", "w", encoding="utf-8") as file:
    json.dump(data, file, ensure_ascii=False, indent=4)

print("\n--- ĐÃ CẬP NHẬT HOÀN TOÀN FILE BANNER.JSON ---")

Dữ liệu đã đọc từ banner.json:
https://cdn2.fptshop.com.vn/unsafe/1240x0/filters:format(webp):quality(75)/H2_614x212_8e2658ed17.png
Đang upload file lên Cloudinary...

--- Upload thành công! ---
Link Cloudinary của bạn: https://res.cloudinary.com/dozpywcus/image/upload/v1780382998/banners_1.webp
Đang upload file lên Cloudinary...

--- Upload thành công! ---
Link Cloudinary của bạn: https://res.cloudinary.com/dozpywcus/image/upload/v1780383000/banners_2.webp
Đang upload file lên Cloudinary...

--- Upload thành công! ---
Link Cloudinary của bạn: https://res.cloudinary.com/dozpywcus/image/upload/v1780383002/banners_3.webp
Đang upload file lên Cloudinary...

--- Upload thành công! ---
Link Cloudinary của bạn: https://res.cloudinary.com/dozpywcus/image/upload/v1780383004/banners_4.webp
Đang upload file lên Cloudinary...

--- Upload thành công! ---
Link Cloudinary của bạn: https://res.cloudinary.com/dozpywcus/image/upload/v1780383006/banners_5.webp
Đang upload file lên Cloudinary...

--- Upl

# import dữ liệu

In [20]:
import json
import mysql.connector
from datetime import datetime, timedelta

In [21]:
db_config = {
    "host": "localhost",
    "port": 3306,
    "database": "phone_store",
    "user": "root",
    "password": "123456",
    "charset": "utf8mb4",  # Đảm bảo lưu đúng font tiếng Việt UTF-8
}

In [23]:
try:
    # Đọc file JSON đã cập nhật link từ Cloudinary
    with open("banner_updated.json", "r", encoding="utf-8") as file:
        json_data = json.load(file)

    # CHỈNH SỬA TẠI ĐÂY: Kiểm tra cấu trúc JSON chuẩn xác
    # Nếu json_data là một Dictionary (chỉ có 1 banner), ta bọc nó vào trong 1 List []
    if isinstance(json_data, dict):
        banners_list = [json_data]
    else:
        # Nếu json_data đã là một List sẵn rồi (nhiều banner) thì giữ nguyên
        banners_list = json_data

    print(f"Tổng số banner thực tế cần import: {len(banners_list)}")

    # Kết nối MySQL
    conn = mysql.connector.connect(**db_config)
    cursor = conn.cursor()

    # Xóa dữ liệu cũ và reset AUTO_INCREMENT
    print("Đang làm sạch dữ liệu cũ trong bảng `banners`...")
    cursor.execute("TRUNCATE TABLE banners;")

    # Câu lệnh Insert SQL
    insert_query = """
    INSERT INTO banners (title, subtitle, image_url, link_url, button_text, position, start_at, end_at, is_active, sort_order)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """

    inserted_count = 0
    # Vòng lặp lấy chỉ mục `i` chạy từ 0
    for i, item in enumerate(banners_list):

        # 1. Logic tính ngày tháng
        now = datetime.now()
        start_date = now.strftime("%Y-%m-%d %H:%M:%S")
        end_date = (now + timedelta(days=100)).strftime("%Y-%m-%d %H:%M:%S")

        # 2. Logic tăng dần số thứ tự (1, 2, 3,...)
        auto_sort_order = i + 1

        val = (
            item.get("title"),
            item.get("subtitle", None),
            item.get("image_url"),
            item.get("link_url", None),
            item.get("button_text", None),
            item.get("position"),
            start_date,
            end_date,
            item.get("is_active", True),
            auto_sort_order,
        )

        cursor.execute(insert_query, val)
        inserted_count += 1
        print(
            f"Đang import: {item.get('title')} | Thứ tự hiển thị: {auto_sort_order}"
        )

    conn.commit()
    print(
        f"\n-> THÀNH CÔNG: Đã import mới hoàn toàn {inserted_count} bản ghi vào bảng banners!"
    )

except Exception as e:
    print(f"Đã xảy ra lỗi hệ thống: {e}")
    if "conn" in locals() and conn.is_connected():
        conn.rollback()
finally:
    if "cursor" in locals() and cursor:
        cursor.close()
    if "conn" in locals() and conn.is_connected():
        conn.close()
        print("Đã đóng kết nối Database an toàn.")

Tổng số banner thực tế cần import: 9
Đang làm sạch dữ liệu cũ trong bảng `banners`...
Đang import: iPhone 17 Pro Max | Thứ tự hiển thị: 1
Đang import: GALAXY S26 ULTRA  Siêu Phẩm AI Galaxy | Thứ tự hiển thị: 2
Đang import: OPPO Find X9 Ultra | Siêu Phẩm Đỉnh Cao | Thứ tự hiển thị: 3
Đang import: GALAXY S25 FE | Siêu Phẩm Mới Của Samsung | Thứ tự hiển thị: 4
Đang import: S26 ULTRA | Siêu Phẩm Mới Thịnh Hành | Thứ tự hiển thị: 5
Đang import: XIAOMI REDMI NOTE 15 | Bền vô đối, giá đáng đồng tiền | Thứ tự hiển thị: 6
Đang import: iPhone 17 | Siêu Phẩm Mới Của Apple | Thứ tự hiển thị: 7
Đang import:  XIAOMI REDMI NOTE 15 PRO | Bền vô đối, giá đáng đồng tiền | Thứ tự hiển thị: 8
Đang import: XIAOMI REDMI NOTE 14 | Giá Hạt Dẻ | Thứ tự hiển thị: 9

-> THÀNH CÔNG: Đã import mới hoàn toàn 9 bản ghi vào bảng banners!
Đã đóng kết nối Database an toàn.
